# LINEBENCH — Notebook 06: Simulation Experiments & Robustness Analysis

**Phase 6 Deliverable (Milestone MS5)**

This notebook demonstrates and visualizes the core empirical results of the simulation study:
1. **E4 Robustness**: Throughput spread across equal-station-count balances and position of heuristic balances.
2. **E5 Secondary Objectives**: Percentile ranking and paired throughput gains of `SO1` and `SO2` vs arbitrary optimal and random balances.
3. **E6 Extra Station**: Total throughput vs station efficiency trade-off ($m$ vs $m+1$) and benefit-to-cost ratio.
4. **E7 Workload Shape**: The "Bowl" phenomenon vs even-load `SO1` under unbuffered and buffered lines.
5. **Paced Synchronous Line**: Station overload probabilities and validation against the analytical normal approximation.
6. **Ablations & Sensitivity**: Lognormal vs Gamma noise invariance and diminishing marginal returns of buffering.

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore', 'FigureCanvasAgg is non-interactive')
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ensure project root is available
project_root = Path.cwd()
if not (project_root / 'src').exists() and (project_root.parent / 'src').exists():
    project_root = project_root.parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

results_dir = project_root / 'results'
print(f"Project root resolved: {project_root}")
print(f"Results directory: {results_dir}")

## 1. E4: Robustness & Equal-Station Throughput Spread

Examining how balances with identical station counts differ in relative throughput ($TH \cdot c$) under variability, and where `HEUR_BEST` falls.

In [ ]:
df_e4_paired = pd.read_csv(results_dir / 'e4_paired_differences.csv')
print("=== E4 Stratified Summary (Bootstrap across instances) ===")
display_cols = ['cv', 'buffer_capacity', 'n_instances', 'rand_spread_mean', 'heur_best_rel_th_mean', 'heur_best_percentile_rank_mean']
print(df_e4_paired[display_cols].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
for b in [0, 1, 3]:
    sub = df_e4_paired[df_e4_paired['buffer_capacity'] == b]
    ax.plot(sub['cv'], sub['rand_spread_mean'] * 100, marker='o', label=f'Buffer b={b}')

ax.set_title('Equal-Station Random Balance Spread vs CV Level')
ax.set_xlabel('Task Time Coefficient of Variation (CV)')
ax.set_ylabel('Mean Spread (% of Design Rate)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 2. E5: Secondary Objectives (SO1 & SO2)

Checking how `SO1` (MinMaxLoad) and `SO2` (Smoothness) perform against arbitrary exact and random balances.

In [ ]:
df_e5 = pd.read_csv(results_dir / 'e5_secondary_objectives_paired.csv')
print("=== E5 Secondary Objectives Performance ===")
e5_cols = ['cv', 'buffer_capacity', 'exact_rel_th_mean', 'so1_rel_th_mean', 'so2_rel_th_mean', 'so1_percentile_rank_mean', 'so2_percentile_rank_mean', 'gain_pct_so1_vs_exact_mean', 'gain_pct_so2_vs_exact_mean']
print(df_e5[e5_cols].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
sub_b0 = df_e5[df_e5['buffer_capacity'] == 0]
sub_b1 = df_e5[df_e5['buffer_capacity'] == 1]

ax.bar(np.array([1, 2, 3]) - 0.15, sub_b0['gain_pct_so1_vs_exact_mean'], width=0.3, label='SO1 Gain vs Exact (b=0)', color='steelblue')
ax.bar(np.array([1, 2, 3]) + 0.15, sub_b1['gain_pct_so2_vs_exact_mean'], width=0.3, label='SO2 Gain vs Exact (b=1)', color='coral')

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['CV=0.10', 'CV=0.25', 'CV=0.50'])
ax.set_ylabel('Mean Throughput Gain over Exact (%)')
ax.set_title('Secondary Objectives: Recovery over Arbitrary Exact Optimum')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 3. E6: Extra Station Trade-off ($m$ vs $m+1$)

Evaluating the trade-off between total line throughput expansion and individual station efficiency.

In [ ]:
df_e6 = pd.read_csv(results_dir / 'e6_extra_station_paired.csv')
print("=== E6 Extra Station Trade-Off ===")
e6_cols = ['cv', 'buffer_capacity', 'rel_th_so1_mean', 'rel_th_m_plus_1_mean', 'pct_gain_total_th_mean', 'pct_change_th_per_stn_mean', 'benefit_cost_ratio_mean']
print(df_e6[e6_cols].round(4).to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 4.5))
sub_e6 = df_e6[df_e6['buffer_capacity'] == 1]

ax1.plot(sub_e6['cv'], sub_e6['pct_gain_total_th_mean'], marker='s', color='green', label='Total Line Throughput Gain (%)')
ax1.set_xlabel('Task Time CV')
ax1.set_ylabel('Total Throughput Gain (%)', color='green')
ax1.tick_params(axis='y', labelcolor='green')

ax2 = ax1.twinx()
ax2.plot(sub_e6['cv'], sub_e6['pct_change_th_per_stn_mean'], marker='^', color='red', linestyle='--', label='Per-Station Productivity Change (%)')
ax2.set_ylabel('Productivity per Station Change (%)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Extra Station (m+1): Total Throughput vs Worker Productivity (Buffer b=1)')
plt.tight_layout()
plt.show()

## 4. E7: Workload Shape — The "Bowl" Phenomenon

Comparing deliberately unbalanced bowl profiles with even-load `SO1` configurations.

In [ ]:
df_e7 = pd.read_csv(results_dir / 'e7_bowl_paired.csv')
print("=== E7 Bowl vs Even Load (SO1) ===")
e7_cols = ['cv', 'buffer_capacity', 'pct_gain_bowl_vs_so1_mean', 'pct_gain_bowl_boot_lo', 'pct_gain_bowl_boot_hi', 'bowl_win_rate_pct']
print(df_e7[e7_cols].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.5))
for b in [0, 1, 3]:
    sub = df_e7[df_e7['buffer_capacity'] == b]
    ax.plot(sub['cv'], sub['pct_gain_bowl_vs_so1_mean'], marker='o', label=f'Buffer b={b}')

ax.axhline(0.0, color='black', linestyle=':', label='SO1 Parity')
ax.set_title('Bowl Profile Gain vs SO1 (Even Load) Across CV and Buffers')
ax.set_xlabel('Task Time CV')
ax.set_ylabel('Throughput Gain over SO1 (%)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Paced Synchronous Lines & Normal Approximation

Evaluating station overload probabilities and the analytical accuracy of the normal approximation (§A11).

In [ ]:
df_paced = pd.read_csv(results_dir / 'e6_paced_summary.csv')
print("=== Paced Line Summary & Normal Approx Verification ===")
print(df_paced[['cv', 'balance_type', 'mean_empirical_line_ovld', 'mean_empirical_max_stn_ovld', 'mae_normal_approx', 'corr_load_ratio_vs_overload']].round(4).to_string(index=False))

df_paced_det = pd.read_csv(results_dir / 'e6_paced_analysis.csv')
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_paced_det['empirical_max_stn_overload_prob'], df_paced_det['norm_approx_max_stn_overload_prob'], alpha=0.3, color='purple')
ax.plot([0, 1], [0, 1], 'r--', label='Perfect Agreement (y=x)')
ax.set_title('Empirical vs Normal Approximation Station Overload Probability')
ax.set_xlabel('Empirical Monte Carlo Overload Probability')
ax.set_ylabel('Normal Approximation (§A11)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Ablations: Noise Distribution & Buffer Diminishing Returns

In [ ]:
df_noise = pd.read_csv(results_dir / 'ablation_noise_distribution.csv')
df_buf = pd.read_csv(results_dir / 'ablation_buffer_sensitivity.csv')

print("=== Noise Ablation: Lognormal vs Gamma Difference (%) ===")
print(df_noise.groupby(['cv', 'buffer_capacity'])['pct_diff_gamma_vs_ln'].agg(['mean', 'std']).round(4))

print("\n=== Buffer Scaling & Marginal Return Ratio ===")
print(df_buf.groupby('cv')[['gain_b0_to_b1_pct', 'gain_b1_to_b3_pct', 'marginal_return_ratio']].mean().round(4))